# Musique NER Masking – Pool Builder (Kaggle)

**Goal:** Apply multiple NER models to Musique pool questions, compare masking quality side-by-side, and export the best masked pool for downstream similarity-based few-shot retrieval.

**NER Models tested:**
1. `dslim/bert-base-NER` (CoNLL-03, base)
2. `dslim/bert-large-NER` (CoNLL-03, large)
3. `FacebookAI/xlm-roberta-large-finetuned-conll03-english` (XLM-R large)
4. `urchade/gliner_medium-v2.1` (GLiNER span-based)

**Placeholder mapping:** PER→`[PERSON]`, ORG→`[ORG]`, LOC→`[LOC]`, MISC→`[MISC]`, other→`[ENTITY]`

In [ ]:
!pip install -q gliner

In [ ]:
import json
import sys
import torch
import transformers
from pathlib import Path
from datetime import datetime

print(f"Python:       {sys.version}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} / {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ── Config ──
SEED = 42
POOL_FILE = Path("/kaggle/input/datasets/jahidahmadistudent/ner-pool/musique_pool_2hop_3hop_4hop.json")
OUTPUT_DIR = Path("/kaggle/working")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
DEVICE = 0 if torch.cuda.is_available() else -1

TYPE_TO_PLACEHOLDER = {
    "PER": "[PERSON]",
    "PERSON": "[PERSON]",
    "ORG": "[ORG]",
    "LOC": "[LOC]",
    "MISC": "[MISC]",
}

NER_MODELS = [
    "dslim/bert-base-NER",
    "dslim/bert-large-NER",
    "FacebookAI/xlm-roberta-large-finetuned-conll03-english",
    "tner/deberta-v3-large-conll2003",
]
GLINER_MODELS = [
    "urchade/gliner_medium-v2.1",
]

print(f"Seed:      {SEED}")
print(f"Device:    {DEVICE}")
print(f"Run ID:    {RUN_ID}")
print(f"Pool file: {POOL_FILE} (exists={POOL_FILE.exists()})")

In [ ]:
# ── Load pool data (single combined file, split by hop_count) ──
raw_pool = json.loads(POOL_FILE.read_text(encoding="utf-8"))
print(f"Loaded {len(raw_pool)} total items from combined pool file")

pool_data: dict[str, list[dict]] = {}
for item in raw_pool:
    hop_key = f"{item['hop_count']}hop"
    pool_data.setdefault(hop_key, []).append(item)

for hop_key in sorted(pool_data):
    print(f"  {hop_key}: {len(pool_data[hop_key])} items")

all_questions = [item["question"] for item in raw_pool]
print(f"\nTotal questions to mask: {len(all_questions)}")

In [ ]:
# ── Masking helpers ──

def apply_spans(text: str, spans: list[tuple[int, int, str]]) -> str:
    """Replace non-overlapping spans sorted by start position."""
    spans = sorted(spans, key=lambda x: x[0])
    merged = []
    last_end = -1
    for s, e, ph in spans:
        if s < last_end:
            continue
        merged.append((s, e, ph))
        last_end = e
    parts = []
    last = 0
    for s, e, ph in merged:
        parts.append(text[last:s])
        parts.append(ph)
        last = e
    parts.append(text[last:])
    return "".join(parts)


def mask_with_hf_pipeline(questions: list[str], model_name: str, device: int) -> list[str]:
    """Mask using HF token-classification pipeline."""
    from transformers import pipeline
    ner = pipeline(
        "token-classification",
        model=model_name,
        tokenizer=model_name,
        aggregation_strategy="max",
        device=device,
    )
    results = []
    for q in questions:
        entities = ner(q)
        spans = []
        for ent in entities:
            s, e = ent.get("start"), ent.get("end")
            if s is None or e is None:
                continue
            label = ent.get("entity_group") or ent.get("entity") or ent.get("label")
            ph = TYPE_TO_PLACEHOLDER.get(str(label), TYPE_TO_PLACEHOLDER.get(str(label).upper(), "[ENTITY]"))
            spans.append((int(s), int(e), ph))
        results.append(apply_spans(q, spans))
    del ner
    torch.cuda.empty_cache()
    return results


def mask_with_manual_inference(questions: list[str], model_name: str, device: int) -> list[str]:
    """Manual inference + BIO decoding — works for DeBERTa and any model
    where the HF pipeline's aggregation is broken."""
    from transformers import AutoTokenizer, AutoModelForTokenClassification

    dev = f"cuda:{device}" if device >= 0 else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)
    model = AutoModelForTokenClassification.from_pretrained(model_name).to(dev)
    model.eval()
    id2label = model.config.id2label

    results = []
    for q in questions:
        enc = tokenizer(q, return_tensors="pt", return_offsets_mapping=True)
        offsets = enc.pop("offset_mapping")[0].tolist()  # list of (start, end)
        inputs = {k: v.to(dev) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**inputs).logits
        preds = logits.argmax(dim=-1)[0].cpu().tolist()

        # BIO → spans
        spans = []
        cur_start, cur_end, cur_tag = None, None, None
        for pred_id, (s, e) in zip(preds, offsets):
            if s == 0 and e == 0:  # special token ([CLS], [SEP], padding)
                if cur_start is not None:
                    spans.append((cur_start, cur_end, cur_tag))
                    cur_start = cur_end = cur_tag = None
                continue

            label = id2label[pred_id]
            if label.startswith("B-"):
                if cur_start is not None:
                    spans.append((cur_start, cur_end, cur_tag))
                cur_tag = label[2:]
                cur_start, cur_end = s, e
            elif label.startswith("I-") and cur_tag is not None and label[2:] == cur_tag:
                cur_end = e
            else:
                if cur_start is not None:
                    spans.append((cur_start, cur_end, cur_tag))
                    cur_start = cur_end = cur_tag = None

        if cur_start is not None:
            spans.append((cur_start, cur_end, cur_tag))

        ph_spans = [
            (s, e, TYPE_TO_PLACEHOLDER.get(tag, TYPE_TO_PLACEHOLDER.get(tag.upper(), "[ENTITY]")))
            for s, e, tag in spans
        ]
        results.append(apply_spans(q, ph_spans))

    del model
    torch.cuda.empty_cache()
    return results


def mask_with_gliner(questions: list[str], model_name: str) -> list[str]:
    """Mask using GLiNER span detector."""
    from gliner import GLiNER
    model = GLiNER.from_pretrained(model_name)
    labels = ["PER", "ORG", "LOC", "MISC"]
    results = []
    for q in questions:
        entities = model.predict_entities(q, labels=labels, threshold=0.5)
        spans = []
        for ent in entities:
            if not isinstance(ent, dict):
                continue
            s = ent.get("start") or ent.get("begin")
            e = ent.get("end") or ent.get("end_idx")
            label = ent.get("label") or ent.get("type")
            if s is None or e is None or label is None:
                continue
            ph = TYPE_TO_PLACEHOLDER.get(str(label).upper(), TYPE_TO_PLACEHOLDER.get(str(label), "[ENTITY]"))
            if int(e) > int(s):
                spans.append((int(s), int(e), ph))
        results.append(apply_spans(q, spans))
    del model
    torch.cuda.empty_cache()
    return results

In [ ]:
# ── Run all NER models ──
masked_results: dict[str, list[str]] = {}

for model_name in NER_MODELS:
    print(f"\n{'='*60}")
    print(f"Running: {model_name}")
    print(f"{'='*60}")
    if "deberta" in model_name.lower():
        masked_results[model_name] = mask_with_manual_inference(all_questions, model_name, DEVICE)
    else:
        masked_results[model_name] = mask_with_hf_pipeline(all_questions, model_name, DEVICE)
    print(f"  Done. Masked {len(masked_results[model_name])} questions.")

for model_name in GLINER_MODELS:
    print(f"\n{'='*60}")
    print(f"Running: {model_name}")
    print(f"{'='*60}")
    masked_results[model_name] = mask_with_gliner(all_questions, model_name)
    print(f"  Done. Masked {len(masked_results[model_name])} questions.")

print(f"\nAll models done. Models: {list(masked_results.keys())}")

In [ ]:
# ── Side-by-side comparison (first 20 questions) ──
import re

def sanitize_slug(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]+", "_", s.split("/")[-1]).strip("_")

N_PREVIEW = 20
model_names = list(masked_results.keys())

for i in range(min(N_PREVIEW, len(all_questions))):
    print(f"\n{'─'*80}")
    print(f"Q{i+1}: {all_questions[i]}")
    for mn in model_names:
        slug = sanitize_slug(mn)
        print(f"  {slug:40s} → {masked_results[mn][i]}")

In [ ]:
# ── Quality metrics: count how many questions got at least one entity masked ──
print(f"{'Model':<55} {'Masked/Total':>15} {'% Masked':>10}")
print("─" * 85)
for mn in model_names:
    masked_count = sum(1 for orig, m in zip(all_questions, masked_results[mn]) if orig != m)
    total = len(all_questions)
    pct = 100.0 * masked_count / total if total else 0
    print(f"{mn:<55} {masked_count:>6}/{total:<6}   {pct:>6.1f}%")

In [ ]:
# ── Placeholder distribution per model ──
import re as _re

placeholder_pat = _re.compile(r"\[(PERSON|ORG|LOC|MISC|ENTITY)\]")

print(f"{'Model':<55} {'[PERSON]':>10} {'[ORG]':>8} {'[LOC]':>8} {'[MISC]':>8} {'[ENTITY]':>10}")
print("─" * 105)
for mn in model_names:
    counts = {"PERSON": 0, "ORG": 0, "LOC": 0, "MISC": 0, "ENTITY": 0}
    for m in masked_results[mn]:
        for match in placeholder_pat.finditer(m):
            tag = match.group(1)
            counts[tag] = counts.get(tag, 0) + 1
    total_tags = sum(counts.values())
    print(f"{mn:<55} {counts['PERSON']:>10} {counts['ORG']:>8} {counts['LOC']:>8} {counts['MISC']:>8} {counts['ENTITY']:>10}  (total: {total_tags})")

In [ ]:
# ── Export masked pool JSON for each model ──

def decomposition_to_string(decomposition) -> str:
    if isinstance(decomposition, str):
        return decomposition.strip()
    if isinstance(decomposition, list):
        steps = [str(s).strip() for s in decomposition if str(s).strip()]
        return "\n".join([f"{i+1}. {s}" for i, s in enumerate(steps)])
    return ""

for mn in model_names:
    slug = sanitize_slug(mn)
    out_path = OUTPUT_DIR / f"musique_few_shot_decompositions_ner_{slug}.json"

    output: dict[str, list[dict]] = {}
    for i, item in enumerate(raw_pool):
        hop_key = f"{item['hop_count']}hop"
        output.setdefault(hop_key, []).append({
            "question": item["question"],
            "masked": masked_results[mn][i],
            "decomposition": decomposition_to_string(item["decomposition"]),
        })

    out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Wrote: {out_path} ({sum(len(v) for v in output.values())} items)")

In [ ]:
# ── Save run artifacts ──
config = {
    "seed": SEED,
    "run_id": RUN_ID,
    "device": DEVICE,
    "ner_models": NER_MODELS,
    "gliner_models": GLINER_MODELS,
    "type_to_placeholder": TYPE_TO_PLACEHOLDER,
    "pool_file": str(POOL_FILE),
    "total_questions": len(all_questions),
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

notes = f"""# NER Masking Run – {RUN_ID}
- Models tested: {len(NER_MODELS)} HF + {len(GLINER_MODELS)} GLiNER
- Total pool questions masked: {len(all_questions)}
- Output files per model in /kaggle/working/
"""
(OUTPUT_DIR / "notes.md").write_text(notes)

print("Artifacts saved.")
print("\nOutput files:")
for f in sorted(OUTPUT_DIR.glob("musique_few_shot_decompositions_ner_*.json")):
    print(f"  {f.name}")